In [1]:
# import the necessary libraries

import ollama
import pandas as pd
import os
import time
from tqdm import tqdm

In [2]:
# Define the paths for the input and output files
input_data_path = "/Users/mariyajose/Documents/Sarcasm_Final/Exploratory_Data_Analysis/final_bitcoin_data.csv"
output_data_path = "/Users/mariyajose/Documents/Sarcasm_Final/Consistency/outputs.csv"

In [3]:
# read the bitcoin data into pandas dataframe bitcoin_data

input_data = pd.read_csv(input_data_path)
input_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 906 entries, 0 to 905
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   label           906 non-null    int64 
 1   comment         906 non-null    object
 2   author          906 non-null    object
 3   subreddit       906 non-null    object
 4   score           906 non-null    int64 
 5   ups             906 non-null    int64 
 6   downs           906 non-null    int64 
 7   date            906 non-null    object
 8   created_utc     906 non-null    object
 9   parent_comment  906 non-null    object
dtypes: int64(4), object(6)
memory usage: 70.9+ KB


In [4]:
if os.path.exists(output_data_path):
    print("Found an existing file with the same name, continuing labeling...")
    new_labeled_df = pd.read_csv(output_data_path)
    
    if 'ds_temp1' not in new_labeled_df.columns:
        new_labeled_df['ds_temp1'] = pd.NA
else:
    print("File not found with the given name, creating a new file and starting labeling from the beginning")
    new_labeled_df = input_data.copy()
    new_labeled_df['ds_temp1'] = pd.NA

# Fixed index finder to handle pd.NA safely
def index_finder(df):
    for index, value in df['ds_temp1'].items():
        if pd.isna(value):
            return index
        if value not in [0, 1, 0.0, 1.0]:
            return index
    return len(df)

start_index = index_finder(new_labeled_df)

if start_index >= len(new_labeled_df):
    print("\n The dataset is already fully labeled. No further labeling is required.")
else:
    print(f"\n Continuing labelling from row index: {start_index} out of {len(new_labeled_df)}")

Found an existing file with the same name, continuing labeling...

 Continuing labelling from row index: 711 out of 906


In [5]:
def labeller(parent_comment, comment):
    prompt = f"""
    Find out if the reply comment is a sarcastic response to the parent comment.
    - The Parent comment: "{parent_comment}"
    - The Reply comment: "{comment}"

    The output should only contain 1 if the reply comment is sarcastic and 0 if the reply comment is nonsarcastic. Do not include any explanation. It should strictly be a 0 or 1.
    """
    try:
        # 1. Swapped the model to deepseek-r1:8b
        result = ollama.chat(
            model='deepseek-r1:8b', 
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 1.0}  # Keeps outputs deterministic
        )
        raw_result = result['message']['content'].strip()
        
        # 2. Extract the first clean number from the model's output
        if "1" in raw_result:
            return 1
        elif "0" in raw_result:
            return 0
        else:
            # Fallback in case it outputs text like "Sarcastic" or "Not sarcastic"
            return 1 if "sarcastic" in raw_result.lower() else 0
            
    except Exception as e:
        raise RuntimeError(f"Ollama Call Error: {e}")

In [6]:
if start_index < len(new_labeled_df):
    print("\nIterating through remaining unlabelled rows via local Ollama instance...")
    start_time = time.time()

    non_labelled_rows = new_labeled_df.iloc[start_index:]

    # Create the progress bar instance
    pbar = tqdm(non_labelled_rows.iterrows(), total=len(non_labelled_rows), miniters=25)

    for index, row in pbar:
        try:
            prediction = labeller(row['parent_comment'], row['comment'])
            new_labeled_df.at[index, 'ds_temp1'] = prediction

            # Update the progress bar text dynamically every 25 rows (without printing new lines)
            if index % 25 == 0:
                pbar.set_description(f"Processing Row {index}")

        except Exception as e:
            # Print errors so you don't miss failure warnings
            tqdm.write(f"Row {index} skipped due to error: {e}")
            continue

        # Save progress every 10 rows
        if index % 10 == 0:
            new_labeled_df.to_csv(output_data_path, index=False)

    # Close progress bar and complete final save
    pbar.close()
    new_labeled_df.to_csv(output_data_path, index=False)

    # Calculate execution metrics
    end_time = time.time()
    total_seconds = end_time - start_time
    hours, minutes, seconds = int(total_seconds // 3600), int((total_seconds % 3600) // 60), int(total_seconds % 60)

    # Final summary display
    print(f"Dataset labeling complete!")
    print(f"Total rows processed: {len(non_labelled_rows)}")
    print(f"Results saved to:     {os.path.abspath(output_data_path)}")
    print(f"Total time elapsed:   {hours}h {minutes}m {seconds}s")



Iterating through remaining unlabelled rows via local Ollama instance...


Processing Row 900: 100%|██████████| 195/195 [57:13<00:00, 17.61s/it] 

Dataset labeling complete!
Total rows processed: 195
Results saved to:     /Users/mariyajose/Documents/Sarcasm_Final/Consistency/outputs.csv
Total time elapsed:   0h 57m 13s
